# Démonstration du Modèle LMTL (Zooplancton)

Ce notebook démontre l'utilisation du nouveau module `seapopym.lmtl` pour simuler la dynamique d'une population de zooplancton (biomasse et production structurée par âge).

In [ ]:
from datetime import timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pint
import xarray as xr

from seapopym.blueprint import Blueprint
from seapopym.controller import SimulationConfig, SimulationController
from seapopym.lmtl.configuration import LMTLParams
from seapopym.lmtl.core import (
    compute_day_length,
    compute_mean_temperature,
    compute_mortality_tendency,
    compute_production_dynamics,
    compute_production_initialization,
    compute_recruitment_age,
)
from seapopym.standard.coordinates import Coordinates


In [ ]:
ureg = pint.get_application_registry()

## 1. Configuration

In [ ]:
# Configuration de la simulation
config = SimulationConfig(
    start_date="2020-01-01",
    end_date="2020-03-01",
    timestep=timedelta(hours=1),
)


# Paramètres avec unités explicites
lmtl_params = LMTLParams(
    day_layer=1,
    night_layer=1,
    tau_r_0=10.38 * ureg.days,  # Sera converti en secondes
    gamma_tau_r=ureg.Quantity(0.00, ureg.degC**-1),
    lambda_0=ureg.Quantity(1 / 10, ureg.day**-1),  # Sera converti en 1/s
    gamma_lambda=ureg.Quantity(0, ureg.degC**-1),
    E=1,  # Sans dimension
    T_ref=ureg.Quantity(0, ureg.degC),
)

## 2. Création de l'Environnement (Forçages)

In [ ]:
times = np.arange(
    config.start_date,
    pd.to_datetime(config.end_date) + config.timestep,
    config.timestep,
    dtype="datetime64[ns]",
)
lats = np.array([0.0])  # Equateur
lons = np.array([0.0])
depths = np.array([10.0, 100.0])
cohorts = (np.arange(0, np.ceil(lmtl_params.tau_r_0.magnitude) + 1) * ureg.day).to("second")
cohorts

In [ ]:
# Température : 20°C partout pour simplifier
temp_data = np.ones((len(times), len(lats), len(lons), len(depths))) * 20.0

temperature = xr.DataArray(
    temp_data,
    coords={
        Coordinates.T.value: times,
        Coordinates.Y.value: lats,
        Coordinates.X.value: lons,
        Coordinates.Z: depths,
    },
    dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value, Coordinates.Z),
    name="temperature",
    attrs={"units": "degC"},
)

# Production Primaire (NPP) : Constante = 1.0 g/m²/day
npp = xr.DataArray(
    np.ones((len(times), len(lats), len(lons))) * 1.0,
    coords={Coordinates.T.value: times, Coordinates.Y.value: lats, Coordinates.X.value: lons},
    dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value),
    name="primary_production",
    attrs={"units": "g/m**2/day"},
)

forcings = xr.Dataset({"temperature": temperature, "primary_production": npp})
forcings


## 3. État Initial

In [ ]:
# Biomasse initiale nulle
biomass_init = xr.DataArray(
    np.zeros((len(lats), len(lons))),
    coords={Coordinates.Y.value: lats, Coordinates.X.value: lons},
    dims=(Coordinates.Y.value, Coordinates.X.value),
    name="biomass",
)

# Production initiale nulle
production_init = xr.DataArray(
    np.zeros((len(lats), len(lons), len(cohorts))),
    coords={Coordinates.Y.value: lats, Coordinates.X.value: lons, "cohort": cohorts},
    dims=(Coordinates.Y.value, Coordinates.X.value, "cohort"),
    name="production",
)

# Variables système (dt en SECONDES, pas en jours!)
dt_val = config.timestep.total_seconds()  # CORRECTION: dt en secondes

# On les ajoute au dataset initial pour qu'ils soient disponibles pour les fonctions
initial_state = xr.Dataset({"biomass": biomass_init, "production": production_init, "dt": dt_val})
initial_state


## 4. Construction du Modèle (Blueprint)

In [ ]:
def configure_model(bp):
    bp.register_forcing(
        "temperature",
        dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value, Coordinates.Z),
        units="degC",
    )
    bp.register_forcing(
        "primary_production",
        dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value),
        units="g/m**2/s",  # CORRECTION: Blueprint attend des unités SI (par seconde)
    )
    bp.register_forcing(
        "production", dims=("cohort", Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value)
    )
    bp.register_forcing(
        "biomass", dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value)
    )

    # Enregistrement des variables système comme forcings/data nodes
    # Elles sont présentes dans l'état initial
    bp.register_forcing("dt")
    bp.register_forcing("cohort")

    # Enregistrement des coordonnées utilisées comme inputs
    bp.register_forcing(Coordinates.T.value)
    bp.register_forcing(Coordinates.Y.value)

    bp.register_group(
        group_prefix="Micronekton",
        units=[
            {
                "func": compute_day_length,
                "output_mapping": {"output": "day_length"},
                "input_mapping": {"latitude": Coordinates.Y.value, "time": Coordinates.T.value},
                "output_units": {"output": "dimensionless"},
            },
            {
                "func": compute_mean_temperature,
                "output_mapping": {"output": "mean_temperature"},
                "output_units": {"output": "degC"},
            },
            {
                "func": compute_recruitment_age,
                "output_mapping": {"output": "recruitment_age"},
                "output_units": {"output": "second"},
            },
            {
                "func": compute_production_initialization,
                "input_mapping": {"cohorts": "cohort"},
                "output_mapping": {"output": "production_source_npp"},
                "output_tendencies": {"output": "production"},
                "output_units": {"output": "g/m**2/second"},
            },
            {
                "func": compute_production_dynamics,
                "input_mapping": {
                    "production": "production",
                    "cohort_ages": "cohort",
                    "dt": "dt",
                },
                "output_mapping": {
                    "production_tendency": "production_tendency",
                    "recruitment_source": "biomass_source",
                },
                "output_tendencies": {
                    "production_tendency": "production",
                    "recruitment_source": "biomass",
                },
                "output_units": {
                    "production_tendency": "g/m**2/second",
                    "recruitment_source": "g/m**2/second",
                },
            },
            {
                "func": compute_mortality_tendency,
                "output_mapping": {"mortality_loss": "biomass_mortality"},
                "output_tendencies": {"mortality_loss": "biomass"},
                "output_units": {"mortality_loss": "g/m**2/second"},
            },
        ],
        parameters={
            "day_layer": {"units": "dimensionless"},
            "night_layer": {"units": "dimensionless"},
            "tau_r_0": {"units": "second"},
            "gamma_tau_r": {"units": "1/degC"},
            "lambda_0": {"units": "1/second"},
            "gamma_lambda": {"units": "1/degC"},
            "T_ref": {"units": "degC"},
            "E": {"units": "dimensionless"},
        },
    )


In [ ]:
bp = Blueprint()
configure_model(bp)
bp.visualize()
plt.show()

## 5. Exécution

In [ ]:
# controller = SimulationController(config, backend="sequential")
# controller.setup(
#     configure_model,
#     initial_state,
#     forcings,
#     parameters={"Micronekton": lmtl_params},
# )
# controller.run()

In [ ]:
from seapopym.standard.coordinates import Coordinates

# Initialisation du contrôleur
controller = SimulationController(config, backend="sequential")
controller.setup(
    configure_model,
    initial_state,
    forcings,
    parameters={"Micronekton": lmtl_params},
)
# Boucle de simulation avec enregistrement de l'historique
print(f"Starting simulation from {config.start_date} to {config.end_date}")
history = []
times = []
current_time = config.start_date
# On enregistre l'état initial (t=0)
# Note: On utilise Coordinates.T s'il est utilisé comme dimension, sinon on l'ajoute
snapshot = controller.state[["biomass", "production"]].copy(deep=True)
snapshot = snapshot.expand_dims({Coordinates.T: [current_time]})
history.append(snapshot)
times.append(current_time)
try:
    while current_time < config.end_date:
        # Exécution d'un pas de temps
        controller.step()
        current_time += config.timestep

        # Capture de l'état
        snapshot = controller.state[["biomass", "production"]].copy(deep=True)
        snapshot = snapshot.expand_dims({Coordinates.T: [current_time]})
        history.append(snapshot)
        times.append(current_time)

    print("Simulation completed.")
except Exception as e:
    print(f"Simulation failed at {current_time}: {e}")
    raise
# Concaténation de l'historique en un seul Dataset
full_history = xr.concat(history, dim=Coordinates.T)


## 6. Visualisation

In [ ]:
# Sélection d'un point spatial (milieu de la grille)
# On utilise les noms de dimensions de l'énumération
y_idx = len(full_history[Coordinates.Y]) // 2
x_idx = len(full_history[Coordinates.X]) // 2
# 1. Évolution de la Biomasse Totale au cours du temps
plt.figure(figsize=(12, 6))
# On somme sur les cohortes (si la dimension s'appelle 'cohort')
total_biomass = full_history["biomass"].isel({Coordinates.Y: y_idx, Coordinates.X: x_idx})
total_biomass.plot(marker="o")
plt.title("Évolution de la Biomasse Totale (un point)")
plt.grid(True)
plt.show()
# 2. Évolution de la Production par Cohorte
plt.figure(figsize=(12, 6))
# On prend quelques cohortes représentatives
cohorts_to_plot = [0, len(full_history.cohort) // 2, len(full_history.cohort) - 1]
for c in cohorts_to_plot:
    prod_c = (
        full_history["production"]
        .isel(cohort=c)
        .isel(
            {
                Coordinates.Y: y_idx,
                Coordinates.X: x_idx,
            },
        )
        .sel({Coordinates.T: slice("2020-01-01", "2020-01-20")})
    )
    prod_c.plot(label=f"Cohorte {c}")
plt.title("Production par Cohorte au cours du temps")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
controller.state